In [1]:
# ==================================================
# Project Two Dashboard - Grazioso Salvare
# CS 499 Milestone Three - Enhanced
# ==================================================
from dash import Dash, dcc, html, dash_table
import dash_leaflet as dl
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
import os

from CRUD_Python_Module import AnimalShelter
import rescue_config
from rescue_config import build_query, get_radio_options, MAX_DISPLAY_ROWS

In [2]:
# Credentials are read from environment variables rather than hardcoded
# in the notebook, which was a finding in the code review.
username = os.environ.get('AAC_USER', 'aacuser')
password = os.environ.get('AAC_PASS')
if not password:
    from getpass import getpass
    password = getpass('MongoDB password: ')

shelter = AnimalShelter(username, password)

In [3]:
# Load only up to MAX_DISPLAY_ROWS documents. The original notebook loaded
# the entire collection at startup, which is O(n) memory.
initial_records = shelter.read({}, page=1, page_size=MAX_DISPLAY_ROWS)
df_initial = pd.DataFrame.from_records(initial_records)

if '_id' in df_initial.columns:
    df_initial.drop(columns=['_id'], inplace=True)

# Ensure numeric types for coordinates and age
df_initial['location_lat'] = pd.to_numeric(
    df_initial['location_lat'], errors='coerce')
df_initial['location_long'] = pd.to_numeric(
    df_initial['location_long'], errors='coerce')
df_initial['age_upon_outcome_in_weeks'] = pd.to_numeric(
    df_initial['age_upon_outcome_in_weeks'], errors='coerce')

# Build the column list once and reuse it (removes the duplicated
# column-definition blocks that appeared in multiple callbacks).
TABLE_COLUMNS = [
    {"name": i, "id": i, "deletable": False, "selectable": True}
    for i in df_initial.columns
]

In [4]:
app = Dash(__name__)

logo_filename = 'Grazioso Salvare Logo.png'
try:
    encoded_logo = base64.b64encode(open(logo_filename, 'rb').read()).decode()
except FileNotFoundError:
    encoded_logo = ''
    print("Logo file not found - dashboard will display without it.")

app.layout = html.Div([
    html.Div(style={'display': 'flex', 'alignItems': 'center',
                    'marginBottom': '20px'}, children=[
        html.A(
            html.Img(src=f'data:image/png;base64,{encoded_logo}',
                     style={'height': '80px', 'marginRight': '20px'}),
            href='https://www.snhu.edu',
            target='_blank'
        ),
        html.H1('SNHU CS-340 Dashboard - Antonio Turrietta')
    ]),
    html.Div("Dashboard created by: Antonio Turrietta",
             style={'fontStyle': 'italic', 'marginBottom': '20px'}),
    html.Hr(),

    html.Div([
        html.Label('Select Rescue Type:', style={'fontWeight': 'bold'}),
        dcc.RadioItems(
            id='filter-type',
            options=get_radio_options(),   # built from rescue_config
            value='reset',
            labelStyle={'display': 'block', 'margin': '5px 0'}
        )
    ], style={'marginBottom': '20px'}),

    # Record-count status line (new): shows how many records match the
    # current filter and how many are displayed.
    html.Div(id='record-count-status',
             style={'fontStyle': 'italic', 'marginBottom': '10px'}),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=TABLE_COLUMNS,
        data=df_initial.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto'},
        style_cell={'minWidth': '100px', 'width': '150px',
                    'maxWidth': '300px', 'whiteSpace': 'normal'},
        style_header={'backgroundColor': 'rgb(230, 230, 230)',
                      'fontWeight': 'bold'}
    ),
    html.Br(),
    html.Hr(),

    html.Div(className='row',
             style={'display': 'flex', 'justifyContent': 'space-around'},
             children=[
                 html.Div(id='graph-id', style={'width': '48%',
                                                'display': 'inline-block'}),
                 html.Div(id='map-id', style={'width': '48%',
                                              'display': 'inline-block'})
             ])
])

In [5]:
# -----------------------------------------------
# Callback 1: Update data table when filter changes
# -----------------------------------------------
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'columns'),
    Input('filter-type', 'value')
)
def update_datatable(filter_type):
    try:
        query = build_query(filter_type)
    except ValueError as e:
        print(f"Invalid filter: {e}")
        return [], TABLE_COLUMNS

    records = shelter.read(query, page=1, page_size=MAX_DISPLAY_ROWS)
    df_filtered = pd.DataFrame.from_records(records)
    if not df_filtered.empty and '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)

    if df_filtered.empty:
        return [], TABLE_COLUMNS

    return df_filtered.to_dict('records'), TABLE_COLUMNS


# -----------------------------------------------
# Callback 2: Update record-count status line
# -----------------------------------------------
@app.callback(
    Output('record-count-status', 'children'),
    Input('filter-type', 'value')
)
def update_record_count(filter_type):
    try:
        query = build_query(filter_type)
    except ValueError:
        return "Invalid filter selected."

    total = shelter.count(query)
    shown = min(total, MAX_DISPLAY_ROWS)
    if total > shown:
        return (f"Showing {shown:,} of {total:,} matching records. "
                f"Refine your filter to see more specific results.")
    return f"Showing {shown:,} matching records."


# -----------------------------------------------
# Callback 3: Pie chart via server-side aggregation
# -----------------------------------------------
@app.callback(
    Output('graph-id', 'children'),
    Input('filter-type', 'value')
)
def update_pie_chart(filter_type):
    try:
        query = build_query(filter_type)
    except ValueError:
        return html.Div("Invalid filter selected.")

    counts = shelter.aggregate_breed_counts(query)
    if not counts:
        return html.Div("No data available for the selected filter.")

    df_counts = pd.DataFrame(counts).rename(columns={'_id': 'breed'})
    fig = px.pie(df_counts, values='count', names='breed',
                 title=f'Breed Distribution - {filter_type.capitalize()} Rescue')
    fig.update_traces(textposition='inside', textinfo='percent+label')
    return dcc.Graph(figure=fig)


# -----------------------------------------------
# Callback 4: Map with O(1) hash-map row lookup
# -----------------------------------------------
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    default_map = dl.Map(style={'width': '100%', 'height': '500px'},
                         center=[30.75, -97.48], zoom=10,
                         children=[dl.TileLayer()])

    if viewData is None or len(viewData) == 0:
        return default_map

    # Build an O(1) lookup dictionary keyed by animal_id.
    lookup = {row.get('animal_id'): row for row in viewData}

    selected_id = None
    if index and len(index) > 0 and index[0] < len(viewData):
        selected_id = viewData[index[0]].get('animal_id')
    if selected_id is None:
        selected_id = viewData[0].get('animal_id')

    row = lookup.get(selected_id)
    if row is None:
        return default_map

    lat = row.get('location_lat')
    lon = row.get('location_long')
    if pd.isna(lat) or pd.isna(lon):
        lat, lon = 30.75, -97.48

    breed = row.get('breed', 'Unknown')
    name = row.get('name', 'Unknown')

    return dl.Map(style={'width': '100%', 'height': '500px'},
                  center=[lat, lon], zoom=12,
                  children=[
                      dl.TileLayer(),
                      dl.Marker(position=[lat, lon],
                                children=[
                                    dl.Tooltip(breed),
                                    dl.Popup([
                                        html.H4(name),
                                        html.P(f"Breed: {breed}")
                                    ])
                                ])
                  ])

In [6]:
# Modern Dash API (works with Dash 4.x). If you are using an older version
# of Dash that supports JupyterDash, replace this with:
#   app.run_server(mode='inline', height=1500)
app.run(jupyter_mode='inline', height=1500)